# Databases & Storage

## What's covered

- **Relational vs NoSQL** stores — what each one is good at, and the four NoSQL families
- **Indexing** — B-tree, hash, and LSM-tree indexes, and what each gives you
- **Replication** — leader-follower, sync vs async, and multi-leader / leaderless setups
- **Sharding** — when to partition writes horizontally, and the common strategies
- **Caching** — where caches sit, the four canonical strategies, and the hard problem of invalidation


## The storage layer

Every non-trivial service has a persistence layer underneath it. The choices made here shape almost every other choice that follows — how the service scales, how it fails, how fast it serves a request, how much it costs to run. The wrong storage decision is one of the most expensive mistakes to reverse, because by the time you discover it you usually have a few terabytes of data and a few hundred services depending on its schema.

Three forces shape the choice:

- **Access pattern.** Do you do lookups by primary key? Range scans? Joins? Aggregations? Full-text search?
- **Consistency requirements.** Does the application need strict transactional guarantees, or can it tolerate eventual consistency in exchange for better availability or latency?
- **Scale shape.** How big is the data? How fast does it grow? Is the bottleneck reads, writes, or storage volume?

The classical split is **relational** (SQL) versus **non-relational** (NoSQL), but that frames the choice too coarsely. Modern systems usually pick a storage engine *per workload* — relational for the transactional core, a search index for full-text, a key-value store for sessions, a time-series store for metrics. We'll walk each family.


## Relational stores — SQL

The relational model: data lives in **tables**, with rows and columns. Rows reference each other by foreign keys. The query language (SQL) lets you join, filter, group, and aggregate declaratively — you say what you want, the engine figures out how.

**What relational stores are good at:**

- **Joins.** Combining data from multiple tables in one query is the relational model's home turf. NoSQL stores can join, but usually awkwardly.
- **ACID transactions.** Atomicity, Consistency, Isolation, Durability — the four guarantees that let business logic treat a sequence of writes as one unit. Either all of it happened, or none of it did.
- **Schema enforcement.** Columns have types. Constraints reject bad data at the boundary. The database refuses to let you write `"abc"` into an integer column.
- **Mature tooling.** Forty years of indexing strategies, query planners, replication tooling, backup tooling.

**Where relational stores hurt:**

- **Horizontal scale of writes.** Sharding a relational database is operationally painful — schema changes have to fan out, cross-shard joins become impossible, ACID can't span shards without distributed transactions.
- **Schema rigidity.** Adding a column to a hundred-million-row table can lock writes for hours unless you have an online-schema-change tool.
- **Flexible documents.** Storing a deeply nested JSON-shaped object in a relational schema means either denormalizing into many tables or using a JSON column (which most relational DBs now support, blurring the line).

**Examples.** PostgreSQL and MySQL are the open-source workhorses. SQL Server and Oracle are the enterprise standards. CockroachDB and Spanner are distributed SQL — they keep the relational model but scale horizontally with consensus-replicated shards.


## Non-relational stores — the four families

"NoSQL" is a misleading umbrella — it covers four very different families. Each one made specific trade-offs to escape a specific limitation of the relational model.

### Key-value stores

The simplest model: a hash map. Set a value at a key, get it back later. No queries, no joins, no schema.

- **Examples:** Redis, Memcached, DynamoDB (at its core), etcd, Consul.
- **Strengths:** sub-millisecond reads, simple horizontal scale (hash the key, route to the shard), trivial to reason about.
- **Use cases:** session storage, caching, leader election (etcd), configuration storage.

### Document stores

Values are structured documents — JSON or BSON, usually with arbitrary nesting. You query by primary key or by indexes on document fields.

- **Examples:** MongoDB, Couchbase, Firestore, DynamoDB (with its document model).
- **Strengths:** schema flexibility (no migration to add a field), natural fit for object-shaped data, horizontal scaling.
- **Use cases:** content management, product catalogs, mobile app backends, anywhere the data shape varies across records or evolves quickly.

### Column-family (wide-column) stores

Data is stored by column rather than by row. A row can have a different set of columns than its neighbours. Optimized for very high write throughput and for queries that touch a few columns across many rows.

- **Examples:** Cassandra, ScyllaDB, HBase, Bigtable.
- **Strengths:** linear horizontal scale of writes, tunable consistency per query, very high throughput.
- **Use cases:** time-series data, event logs, analytics ingestion, very-high-write workloads where the cost of a relational system would be prohibitive.

### Graph stores

Data is nodes and edges. Queries traverse the graph rather than joining tables.

- **Examples:** Neo4j, Amazon Neptune, TigerGraph, JanusGraph.
- **Strengths:** multi-hop traversals are cheap (the graph is the index); naturally expresses relationships.
- **Use cases:** social networks, recommendation engines, fraud detection, anywhere "who is connected to whom" is the central question.


## SQL vs NoSQL — picking

A practical decision framework rather than a religious war:

| Question | Suggested family |
|---|---|
| "I have rich relationships between entities and need to join across them." | **Relational** |
| "I need ACID transactions across multiple records." | **Relational** (or distributed SQL like Spanner/CockroachDB) |
| "My data is naturally a tree of nested fields, and the shape varies per record." | **Document** |
| "I need sub-millisecond reads on simple lookups, at any scale." | **Key-value** |
| "I'm writing tens of thousands of records per second and rarely deleting." | **Column-family** |
| "My queries are about traversing relationships, not aggregating values." | **Graph** |
| "I need full-text search with ranking." | **Specialized search index** (Elasticsearch, OpenSearch) |
| "I have time-series metrics with high write volume and time-based aggregation." | **Time-series DB** (InfluxDB, TimescaleDB, Prometheus) |

The honest answer for most early-stage applications: **start with a relational store**. It's the most flexible default. The data model can evolve as you learn what the application actually does. Move *parts* of the workload to a specialized store when the relational store is provably the bottleneck — not before.

The danger of over-specializing too early: you end up with five storage systems for fifty-thousand requests per second, each with its own operational burden, and none of them under serious load. Better one boring database under control than five exotic ones in flames.


## Indexes — why queries don't scan everything

Without an index, finding `SELECT * FROM users WHERE email = 'x'` requires reading every row in the table — a *full table scan*. For a billion-row table, that's terabytes of I/O for one query.

An **index** is a separate data structure that maps from a search key to row locations, letting the database jump straight to the rows it needs. The trade-off: indexes cost storage and slow down writes (every insert/update touches the index too). Read speed is bought with write cost and disk space.

Three index structures dominate.

### B-tree (and B+tree)

The classical relational-database index. A balanced tree where each node holds a sorted range of keys and pointers to children. Lookups walk from root to leaf in **O(log n)** — typically 3-5 levels deep even for billion-row tables, because each node fans out 100-500 children.

- **Strengths:** range scans (`WHERE x BETWEEN 10 AND 20`), ordered iteration, prefix matching.
- **Cost:** modest write amplification — each insert rebalances at most a few nodes.
- **Used by:** PostgreSQL, MySQL InnoDB, SQL Server, virtually every relational engine.

### Hash index

A hash table from key to row location. **O(1)** lookups, but no order — range scans don't work.

- **Strengths:** equality lookups are even faster than B-tree, especially in memory.
- **Cost:** no range scans, no prefix matching, susceptible to rebuild on resize.
- **Used by:** in-memory stores (Redis), some specialized indexes in relational databases.

### LSM tree (log-structured merge tree)

A write-optimized structure. New writes land in an in-memory buffer; periodically the buffer is sorted and flushed to disk as an *SSTable* (sorted string table). Multiple SSTables accumulate, and a background **compaction** process merges them into fewer, larger files.

- **Strengths:** writes are *sequential* (append-only), which is enormously faster than random writes on SSDs and HDDs. High write throughput.
- **Cost:** reads may have to check several SSTables (mitigated by bloom filters). Compaction consumes CPU and I/O continuously. Tail-latency spikes during major compactions.
- **Used by:** Cassandra, RocksDB, LevelDB, Bigtable, HBase, BigQuery's storage, modern column-family stores.

**The pattern.** If your workload is read-heavy with mixed equality and range queries, B-tree wins. If it's equality-only and latency-critical, hash. If it's write-heavy and you can absorb compaction cost in the background, LSM. Most real systems use multiple index structures for different workloads.


## Replication — multiple copies of the data

A single database server has two problems: it's a single point of failure, and it's a single point of capacity for reads. **Replication** copies the data to multiple servers, giving you both fault tolerance and read scale.

### Leader-follower (primary-replica)

The dominant pattern. One node is the **leader** (also called primary or master). All writes go to it. Writes are then replicated to one or more **followers** (replicas or secondaries). Reads can go to the leader or to any follower.

```
                  +--------+
   writes -----> | leader |  -- replication --> follower 1 --,
                  +--------+                                  |--> reads
                       |                  -> follower 2 ----,'
                       +--reads---------> (optional)
```

The two questions to ask about any leader-follower setup:

**1. Synchronous or asynchronous replication?**

- **Synchronous** — the leader waits for at least one follower to acknowledge the write before returning success. The follower is guaranteed to have the write. If the leader fails immediately after, no data is lost. But the write latency includes the round-trip to the follower, and if the follower is slow or down, writes block.
- **Asynchronous** — the leader returns success immediately and ships the write to followers in the background. Fast and resilient. But if the leader fails before a write reaches the followers, that write is lost.

Real systems often use **semi-synchronous** — wait for at least one follower, but no more — to balance the two.

**2. How does failover work?**

When the leader fails, one of the followers must become the new leader. This requires (a) detecting the failure quickly without false positives, (b) picking the most up-to-date follower, (c) reconfiguring clients to talk to the new leader. Modern systems automate this with consensus protocols (Raft / Paxos — covered in notebook 04). Hand-rolled failover is a frequent source of split-brain incidents.

### Multi-leader and leaderless

Two patterns that drop the single leader.

**Multi-leader** — multiple nodes accept writes. Each leader replicates to the others. Used in multi-region setups where each region has its own leader to keep write latency low. The hard problem: **conflicts**. Two regions write to the same row at the same time — which wins? Solutions range from last-write-wins (lossy) to CRDTs (mathematically merge) to application-level conflict resolution (the application picks).

**Leaderless** — any node can accept any write or read. Writes are sent to N nodes; reads query N nodes and reconcile. The Dynamo paper introduced this style and Cassandra, ScyllaDB, and Riak follow it. Tunable consistency: pick how many writes (`W`) and reads (`R`) must succeed, and as long as `W + R > N`, you get a guaranteed read of the latest write.

The trade-offs around all of these — what's lost, what's preserved, what's eventually consistent — are the subject of notebook 04 (CAP theorem, consistency models). For now: **single-leader, semi-sync, automated failover** is the default for most workloads, and you should only move off it when you have a specific reason.


## Sharding — splitting data across machines

Replication scales reads but not writes — every write still goes to the single leader. To scale writes, you have to split the data itself across multiple machines. That's **sharding** (also called **horizontal partitioning**).

Each shard holds a subset of rows. A row's shard is decided by its **partition key**. Queries routed by partition key go straight to the relevant shard; queries that span partition keys touch multiple shards and have to aggregate.

**Three sharding strategies.**

### Range-based sharding

Partition key ranges map to shards. Shard 1 holds keys `a-h`, shard 2 holds `i-p`, shard 3 holds `q-z`.

- **Strengths:** range scans are localized — scanning `WHERE key BETWEEN 'c' AND 'e'` touches one shard.
- **Weakness:** hot spots. If keys cluster (timestamps, sequential IDs), one shard takes all the writes. The classic disaster pattern.
- **Used by:** HBase, MongoDB (range-sharded option), Bigtable.

### Hash-based sharding

Hash the partition key, take modulo or use consistent hashing, route to that shard. Distribution is uniform regardless of key skew.

- **Strengths:** even load distribution.
- **Weakness:** range scans require touching every shard.
- **Used by:** Cassandra, DynamoDB (default), most key-value stores.

### Geo-based sharding

Shard by user region — EU users on EU shards, US users on US shards. Reduces cross-region latency and helps with data-residency regulations (GDPR).

- **Strengths:** local reads and writes are fast, regulatory alignment.
- **Weakness:** uneven shard sizes (regions aren't equally popular), cross-region operations (a user moving regions, a global aggregation) become expensive.
- **Used by:** most globally distributed user-facing services.

**The hardest problem in sharding is resharding.** When a shard gets too big, you have to split it — moving data without downtime, updating routing without race conditions, keeping clients pointed at the right place. Consistent hashing (from notebook 02) makes this much less painful by only moving 1/N of keys per shard change. Modern sharded systems also use **virtual shards** — many more logical shards than physical machines — so growing the cluster is a remapping of virtual-to-physical, not a data move.

**The honest progression.** Start single-node. When that hurts, add replication for read scale. When writes hurt, shard. Each step adds operational complexity; don't take them speculatively.


## Caching — where it sits and what it costs

A cache is a fast, lossy store in front of a slower authoritative store. The cache hit is cheap (microseconds); the cache miss falls back to the authoritative store (milliseconds) and writes the value back.

**Where caches sit:**

- **Browser cache** — the user's machine. Free; controlled by HTTP headers.
- **CDN cache** — edge POPs. Covered in notebook 02.
- **Reverse-proxy cache** — Varnish, Nginx in front of your app. Caches whole HTTP responses keyed by URL.
- **In-process cache** — a hash map in your application's memory. Fastest possible; per-server, so not shared.
- **Distributed cache** — Redis, Memcached. Shared across your fleet, sub-millisecond reads.
- **Database query cache** — the database remembers the result of recent queries. Less common today; most engines have backed off this in favour of plan caches.

**The four canonical caching strategies** for the application-to-cache-to-database path:


### The four caching strategies


In [ ]:
# Demonstrate four caching strategies against a simulated slow database.

import time
from dataclasses import dataclass, field


@dataclass
class FakeDb:
    store: dict[str, str] = field(default_factory=dict)
    reads: int = 0
    writes: int = 0

    def get(self, key: str) -> str | None:
        self.reads += 1
        time.sleep(0.001)   # simulate 1ms DB read
        return self.store.get(key)

    def put(self, key: str, value: str) -> None:
        self.writes += 1
        time.sleep(0.001)
        self.store[key] = value


@dataclass
class FakeCache:
    store: dict[str, str] = field(default_factory=dict)
    hits: int = 0
    misses: int = 0

    def get(self, key: str) -> str | None:
        if key in self.store:
            self.hits += 1
            return self.store[key]
        self.misses += 1
        return None

    def put(self, key: str, value: str) -> None:
        self.store[key] = value

    def invalidate(self, key: str) -> None:
        self.store.pop(key, None)


# Strategy 1 — Cache-aside (lazy loading).
# The application is in charge. On read, check cache; on miss, load from DB
# and populate the cache. On write, update the DB and invalidate the cache.
class CacheAside:
    def __init__(self, db, cache):
        self.db, self.cache = db, cache

    def get(self, key):
        v = self.cache.get(key)
        if v is None:
            v = self.db.get(key)
            if v is not None:
                self.cache.put(key, v)
        return v

    def put(self, key, value):
        self.db.put(key, value)
        self.cache.invalidate(key)


# Strategy 2 — Read-through. The cache wraps the DB; the app only talks to
# the cache. On miss, the cache itself fetches and populates.
class ReadThroughCache:
    def __init__(self, db, cache):
        self.db, self.cache = db, cache

    def get(self, key):
        v = self.cache.get(key)
        if v is None:
            v = self.db.get(key)
            if v is not None:
                self.cache.put(key, v)
        return v


# Strategy 3 — Write-through. On write, the app writes to the cache; the
# cache synchronously writes to the DB. Cache and DB stay in lockstep.
class WriteThrough:
    def __init__(self, db, cache):
        self.db, self.cache = db, cache

    def put(self, key, value):
        self.cache.put(key, value)
        self.db.put(key, value)        # synchronous — slow but consistent


# Strategy 4 — Write-behind (write-back). The app writes to the cache only;
# the cache flushes to the DB asynchronously. Very fast writes, but data
# can be lost if the cache fails before flushing.
class WriteBehind:
    def __init__(self, db, cache):
        self.db, self.cache = db, cache
        self.pending: list[tuple[str, str]] = []

    def put(self, key, value):
        self.cache.put(key, value)
        self.pending.append((key, value))   # in real systems, a background flusher drains this

    def flush(self):
        for k, v in self.pending:
            self.db.put(k, v)
        self.pending.clear()


# Compare cache-aside read behaviour: first miss, then hits.
db = FakeDb(store={"u:1": "Alice", "u:2": "Bob"})
cache = FakeCache()
svc = CacheAside(db, cache)

for _ in range(5):
    svc.get("u:1")

print(f"DB reads: {db.reads}  (only the first one missed)")
print(f"cache:    hits={cache.hits}, misses={cache.misses}")

# Write-behind buys fast writes at the cost of an unflushed buffer.
db, cache = FakeDb(), FakeCache()
wb = WriteBehind(db, cache)
for i in range(100):
    wb.put(f"k{i}", str(i))
print(f"\nafter 100 writes via write-behind:")
print(f"  DB writes so far:  {db.writes}")
print(f"  pending in cache:  {len(wb.pending)}")
wb.flush()
print(f"  DB writes after flush: {db.writes}")


**Picking a strategy.**

- **Cache-aside** — the default for most applications. Simple, application is in control, easy to add to existing code.
- **Read-through** — when you want a cleaner abstraction and the cache library supports it (Redis modules, some ORM-level caches).
- **Write-through** — when consistency between cache and DB matters more than write latency, and you can absorb the synchronous DB write.
- **Write-behind** — when write throughput matters more than durability, and you can tolerate losing writes that haven't flushed yet. Used in metrics/analytics pipelines, not in transactional systems.


## Cache invalidation — the hard problem

Phil Karlton's quip — "there are only two hard things in computer science: cache invalidation and naming things" — is half a joke and half deadly serious. Stale caches cause some of the worst bugs in production: data that should have changed didn't, transactions appear to roll back, users see other users' data.

**Three approaches to keeping caches fresh:**

- **TTL expiry** — every cached entry has a time-to-live. After the TTL, the entry is treated as missing and re-fetched. Simple. Stale data lives for at most TTL seconds — which may be seconds or hours depending on what tradeoff you want.
- **Explicit invalidation** — on every write that changes the underlying data, send a delete to the cache for the affected keys. Fast freshness. Easy to forget for a write path you didn't think of, which means the cache silently goes stale and you don't know.
- **Versioned keys** — change the cache key when the underlying data changes. `user:42:v17` instead of `user:42`. Old keys are never re-read (no one knows their name); new keys are populated on demand. The same trick CDNs use for asset versioning.

**Cache invalidation across services** is harder. If service A writes to the database and service B has a cache of derived data, B needs to know to invalidate. Three mechanisms: (a) **change-data-capture** — A's database emits a stream of changes, B subscribes; (b) **event bus** — A publishes a "user updated" event, B subscribes; (c) **shared invalidation channel** — A explicitly publishes a cache invalidation message. We come back to all three in notebook 05.

**The pragmatic rule.** If staleness for a few seconds is acceptable, prefer TTL — it's the simplest mental model and the least error-prone. Use explicit invalidation only when the freshness budget is tighter than a TTL allows, and pair it with monitoring on cache hit rates and on data divergence between cache and source.


Notebook four turns to **Distributed Systems Theory** — the conceptual core of the series. The cap theorem and what it really says (and what it doesn't). The consistency models that real systems actually offer — strong, eventual, causal. The consensus protocols — raft and paxos — that let nodes agree on a single answer. Lamport clocks and vector clocks for reasoning about order without a shared wall clock. And leader election. After this notebook, the trade-offs in replication, sharding, and messaging all come from one consistent vocabulary.
